# 🚗 Практикум: рынок авто Lalafo

### Линейная алгебра и статистика на живых — и очень грязных — данных

Перед вами **54 518 реальных объявлений** о продаже машин с доски Lalafo (Кыргызстан).
Данные настоящие, а значит — грязные: цены то в сомах, то в долларах (и всё в одной колонке!),
пробег «1 111 111 111 км», год выпуска 1920, автосалоны с двумя сотнями объявлений и цена «1 сом».

Это не баг, это **самое интересное**. Именно на такой грязи видно разницу между тем,
- как на цифры смотрит **обыватель** («ну цена и цена, чего думать»), и
- что видит **исследователь** («секунду, а половина этих цен — вообще в долларах»).

Ваша задача в каждом кейсе — не «посчитать среднее», а **принять решение и защитить его графиками**.

## 🎯 Правила игры

1. **Пять кейсов** (T1…T5). Каждый — житейское решение, где на кону деньги.
2. В начале кейса **запишите свой наивный ответ** (одна строка, честно, до анализа). Потом проверите, обманула ли вас интуиция.
3. Стройте **много графиков**. График, который меняет ваше мнение, — это удача, а не поражение.
4. Финал каждого кейса — **вердикт**: одно решение + 3–5 графиков, которые его доказывают.
5. В конце — **защита**: 3 минуты у доски. Не пересказ формул, а «я советую вот это, и вот почему».

**Две планки в каждом кейсе:**
- 🟢 *Минимум* — довести до вердикта хотя бы по основному признаку.
- 🔵 *Если зашло* — копнуть глубже (доп. признаки, спорные случаи, красивая визуализация).

> Загрузка данных и черновая обработка признаков **уже сделаны за вас** (ячейки ниже).
> Не тратьте время на парсинг строк — тратьте его на размышление.

---
## ⚙️ Шаг 0. Настройка окружения

Просто запустите. Ставить ничего не нужно — всё есть в Colab.

In [ ]:
import numpy as np, pandas as pd, re
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams.update({"figure.figsize": (8, 5), "figure.dpi": 110,
                     "axes.titlesize": 13, "axes.grid": True})
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)
print("Окружение готово.")

## 📥 Шаг 1. Загрузка датасета

Берём **только таблицы объявлений и продавцов** — фотографии (≈78 ГБ!) не качаем, они не нужны.

In [ ]:
from datasets import load_dataset

REPO = "aiacademy-kg/lalafo-kg-cars"
# config "listings" = ads, "users" = sellers. We never touch config "images".
raw   = load_dataset(REPO, "listings", split="train").to_pandas()
users = load_dataset(REPO, "users",    split="train").to_pandas()
print("Объявлений:", len(raw), "| продавцов:", len(users))

## 🧹 Шаг 2. Черновой инжиниринг признаков (готово за вас)

Здесь мы чиним **типы данных** (год, пробег, объём двигателя — это строки!) и приводим цену к
одной валюте. А вот решение, что считать **выбросом/мусором** в ценах и пробеге, мы намеренно
оставляем вам — это часть кейсов.

Прочитайте комментарии: полезно понимать, откуда взялась каждая новая колонка.

In [ ]:
# --- Rough feature engineering: fix TYPES + unify currency only. -------------
# We deliberately DO NOT drop price/mileage outliers here: deciding "garbage vs
# real" is YOUR job in the cases below.

USD_KGS  = 89.0    # approx 2024-2025 rate; good enough for rough comparison
REF_YEAR = 2026    # to compute car age

def _first_num(x):
    """First number out of a messy string like '2.5 л' or '155 000'."""
    if not isinstance(x, str):
        return np.nan
    m = re.search(r"(\d+(?:[.,]\d+)?)", x)
    return float(m.group(1).replace(",", ".")) if m else np.nan

def _year(x):
    if not isinstance(x, str):
        return np.nan
    m = re.search(r"(\d{4})", x)
    return float(m.group(1)) if m else np.nan

_EMOJI = re.compile(r"[\U0001F000-\U0001FAFF☀-➿]")

def engineer(listings, users):
    df = listings.copy()

    # 1) price in ONE currency (KGS). ~40% of ads are in USD! Unknown -> NaN.
    price = df["price"].astype(float)
    df["price_kgs"] = np.select(
        [df["currency"].eq("KGS"), df["currency"].eq("USD")],
        [price, price * USD_KGS],
        default=np.nan,
    )

    # 2) specs: strings -> numbers (garbage LEFT IN on purpose)
    df["year_n"]    = df["year"].map(_year)
    df["mileage_n"] = df["mileage_km"].map(_first_num)
    df["engine_n"]  = df["engine_volume_l"].map(_first_num)
    df["car_age"]   = REF_YEAR - df["year_n"]

    # 3) handy flags
    df["is_ev"]     = df["fuel"].fillna("").str.contains("Электро")
    df["is_import"] = df["registration_country"].notna() & df["registration_country"].ne("Кыргызстан")

    # 4) engagement ratios (guard against divide-by-zero)
    v = df["views"].replace(0, np.nan)
    df["conv_rate"] = df["callers_count"] / v        # calls per view
    df["fav_rate"]  = df["favorite_count"] / v       # favorites per view

    # 5) seller signals
    df["n_ads_in_data"] = df.groupby("user_id")["ad_id"].transform("size")
    u = users.drop_duplicates("user_id").set_index("user_id")
    df["user_ads_count"]       = df["user_id"].map(u["ads_count"])
    df["seller_response_rate"] = df["user_id"].map(u["response_rate"])
    df["seller_pro"]           = df["user_id"].map(u["pro"])

    # 6) cheap text signals
    t = df["title"].fillna("")
    df["title_len"]       = t.str.len()
    df["title_has_emoji"] = t.map(lambda s: bool(_EMOJI.search(s)))  # Python re, backend-safe
    return df

df = engineer(raw, users)
print("Готово:", df.shape[0], "объявлений,", df.shape[1], "колонок")
df[["title", "price_kgs", "currency", "brand", "year_n", "mileage_n",
    "engine_n", "n_ads_in_data"]].head()

## 📖 Шаг 3. Обзор датасета: что за признаки и что они значат

Одна строка = одно объявление. Ниже — самые полезные колонки. ⭐ помечены те, что мы **добавили сами**.

### Про машину
| Колонка | Что это | Подвох |
|---|---|---|
| `title`, `description` | заголовок и текст | заголовок **генерируется из характеристик** → у разных машин совпадает |
| `brand`, `model` | марка и модель | 108 марок; Toyota — самая частая |
| `year` | год выпуска (**строка!**) | есть 1920 и опечатки |
| ⭐ `year_n`, `car_age` | год числом и возраст (2026 − год) | |
| `mileage_km` | пробег (**строка!**) | пусто у 48%; мусор до 1.1 млрд км; бывает вписан в тысячах |
| ⭐ `mileage_n` | пробег числом | **выбросы не убраны** — это ваша работа |
| `engine_volume_l` | объём двигателя (**строка!**) | ⭐ `engine_n` — числом |
| `fuel`, `transmission`, `body_type`, `drive` | бензин/дизель/газ/гибрид/электро; коробка; кузов; привод | бывают комбо: «Бензин, Газ» |
| `condition`, `technical_condition` | Новый/Б-у; Идеальное/Хорошее/**На запчасти**/Аварийное | много пропусков |
| `customs_cleared`, `registration_country` | растаможен ли; страна учёта | пусто у 52% / 16% |
| `battery_capacity` | ёмкость батареи | ⚠️ есть **только у электро/гибридов** (94% пусто) |

### Цена / внимание / продавец
| Колонка | Что это | Подвох |
|---|---|---|
| `price`, `currency` | цена и валюта как в объявлении | **40% цен в USD, 60% в KGS — в одной колонке!** |
| ⭐ `price_kgs` | цена, приведённая к сомам (USD×89) | выбросы (1 … 1.6 млрд) **не убраны** |
| `views`, `favorite_count`, `callers_count` | просмотры / в избранное / позвонили | сильный разброс |
| ⭐ `conv_rate`, `fav_rate` | звонки-на-просмотр, избранное-на-просмотр | «эффективность» объявления |
| `user_id`, `is_vip`, `is_premium` | продавец и платное продвижение | у салона бывает 190+ объявлений |
| ⭐ `n_ads_in_data`, `user_ads_count` | сколько объявлений у продавца | большое число = автосалон/перекуп |
| ⭐ `seller_response_rate`, `is_import` | как быстро отвечает; пригнана ли из-за рубежа | |

> 🔎 **Две легенды для затравки.**
> 1) Поле `price` смешивает **сомы и доллары**. Пока не приведёте валюту к одной, любое
>    «среднее по цене» — это сложение метров с килограммами (см. T1).
> 2) Кто-то указал пробег `1 111 111 111` км и цену `1 сом`. Мы это **не** трогали —
>    найдёте и обезвредите сами (T1, T5). 😉

## 🏋️ Разминка (не оценивается)

Готовый **шаблон** «сгруппировать → посчитать → нарисовать». Скопируйте и переделывайте под свои задачи.

In [ ]:
# WARM-UP TEMPLATE: how many ads per brand? (groupby -> count -> barplot)
top = df["brand"].value_counts().head(12)

plt.figure(figsize=(8, 5))
sns.barplot(x=top.values, y=top.index, color="#4c72b0")
plt.xlabel("Число объявлений"); plt.ylabel("")
plt.title("Топ-12 марок на доске")
plt.tight_layout(); plt.show()

---
# 🧩 T1. Новее — дешевле?!

> Ты листаешь доску, сортируешь по году и замечаешь странность: некоторые машины 2024 года стоят дешевле, чем ровесники из 2010-го. Доска сошла с ума — или ты чего-то не видишь?

**🙂 Наивный взгляд.** Цена есть цена. Посчитаю среднюю по годам — и увижу, как машины дорожают год к году.

**🔬 А что видит исследователь.** На доске **40% цен — в долларах**, а остальные в сомах, и всё свалено в одну колонку `price`. Складывать доллары с сомами — то же самое, что метры с килограммами. Дорогие свежие авто выставляют в USD, поэтому в «сомовой» части остаются в основном дешёвые — вот и мираж «новые дешевле». Плюс в ценах живут выбросы (от 1 сома до 1.6 млрд), которые добивают среднее.

### Ваша задача
- Воспроизведите «парадокс»: медиана цены по годам на **сырой** цене (только сомы) — где абсурд?
- Приведите валюту к одной (`price_kgs` уже готов) и постройте лестницу заново.
- Отсеките мусорные цены (обоснуйте порог) и покажите честную «типичную цену» по годам.
- Вердикт: как на самом деле цена зависит от года выпуска.

### 🧮 Какой матаппарат стоит попробовать (и зачем именно он)
- **Приведение валюты к одной шкале.** Без этого любая статистика по `price` — бессмыслица (складываем $ и сом). Это первый и главный шаг: сравните, что даёт `price` и что `price_kgs`.
- **Медиана вместо среднего.** Среднее раздуто выбросами (по сомам ≈480k против медианы 300k) и мусором до 1.6 млрд — оно врёт. Медиана устойчива.
- **Боксплот + правило 1.5·IQR.** Чтобы *увидеть* разброс и формально отделить мусорные цены.
- **Перцентили.** Отрезать 1% с краёв — быстрый способ убрать «1 сом» и «1.6 млрд».

### 📊 Что построить
- Две лестницы «медиана цены по году» рядом: по сырой `price` (только KGS) и по `price_kgs`. Мираж и правда.
- Боксплот `price_kgs` в разумных пределах — где выбросы.
- Гистограмма `price_kgs` в **log-шкале** (сильный скос).

### 🎤 Как защищать
Покажите две лестницы рядом: «что кажется» и «что на самом деле». Назовите честной цифрой, во сколько раз свежая машина дороже десятилетней.

*Прежде чем писать код — впишите сюда свой наивный ответ одной строкой:* `...`

In [ ]:
# --- T1: ваш код. План — в задании выше. ------------------------------------
# Подсказка по шагам:
#   1) наивно: медиана СЫРОЙ цены по году, но только где валюта KGS
#      raw = df[(df.currency=="KGS") & df.price.notna()]
#      raw.groupby(raw.year_n.astype("Int64")).price.median()   # где абсурд с 2024?
#   2) честно: то же по price_kgs (валюта уже приведена USD->сом)
#   3) отсеките мусор (свой порог!) и постройте boxplot price_kgs
#   4) сравните две лестницы -> вердикт

---
# 🧩 T2. Пробег решает?

> Один продавец: «У меня маленький пробег, поэтому дороже — бери!». Сосед по двору: «Да брось, возьми с пробегом побольше и дешевле, движок тот же». Кто из них прав?

**🙂 Наивный взгляд.** Чем меньше пробег — тем дороже машина. Пробег — главный ценник, на него и смотрим.

**🔬 А что видит исследователь.** Если построить медиану цены по пробегу, картинка не монотонная: самые малопробежные машины порой **дешевле** среднепробежных, а не дороже. Причина — пробег «тонет» в марке и годе: при **одинаковом** пробеге ~170 000 км Toyota стоит ≈1.37 млн, а Daewoo ≈240 тыс. (разница в шесть раз!). Пробег играет **внутри** одной модели, а между разными машинами его перебивают марка и возраст.

### Ваша задача
- Проверьте наивную связь «меньше пробег → дороже» по всей доске.
- Разложите по маркам: сохраняется ли она внутри марки?
- Зафиксируйте одну модель и год — виден ли эффект пробега там?
- Ответьте, кто прав: продавец или сосед.

### 🧮 Какой матаппарат стоит попробовать (и зачем именно он)
- **Медиана цены по бакетам пробега (`groupby` + `pd.cut`).** Наивная картинка — с неё начать, чтобы оспорить.
- **2D-KDE (пробег, цена) с раскраской по марке + центроиды.** Нужно, чтобы увидеть: облака марок разъезжаются по цене, а пробег играет уже *внутри* облака. Центроид = (средний пробег, средняя цена) марки.
- **Фиксируем марку+год** и смотрим пробег внутри (например, Toyota Camry одного года).
- **z-стандартизация.** Пробег (сотни тысяч) и цена (миллионы) — разные шкалы; приведите к сопоставимым. ⚠️ Сначала уберите мусорный пробег (>1 млн км) — иначе шкала сломается.

### 📊 Что построить
- Scatter `mileage_n`–`price_kgs`, раскрашенный по марке.
- 2D-KDE (или scatter) с **центроидами** 2–3 марок.
- Медиана цены по пробегу **внутри** одной модели (напр. Toyota Camry).

### 🎤 Как защищать
Один график, где видно: по всей доске пробег почти ничего не решает, а внутри одной модели — решает. Дайте покупателю чёткий вердикт.

*Прежде чем писать код — впишите сюда свой наивный ответ одной строкой:* `...`

In [ ]:
# --- T2: ваш код. План — в задании выше. ------------------------------------
# Подсказка по шагам:
#   1) sub = df[df.price_kgs.notna() & df.mileage_n.between(1, 500000)]  # уберите мусор
#   2) наивно: медиана цены по бакетам пробега
#      sub["mb"] = pd.cut(sub.mileage_n, [0,50000,100000,150000,200000,300000,500000])
#      sub.groupby("mb", observed=True).price_kgs.median()   # монотонно падает? или сюрприз на малом пробеге?
#   3) 2D: sns.scatterplot(sub, x="mileage_n", y="price_kgs", hue="brand")
#      центроиды: sub.groupby("brand")[["mileage_n","price_kgs"]].mean()
#   4) зафиксируйте одну модель (напр. Toyota Camry) и посмотрите пробег внутри неё

---
# 🧩 T3. Найди альтернативу этой машине

> Ты присмотрел Toyota Camry 2015, 2.5 л, автомат — но её только что продали. Хочешь «что-то похожее». Как найти похожие машины, не листая доску вручную?

**🙂 Наивный взгляд.** Искать только Toyota Camry ровно того же года — и ждать.

**🔬 А что видит исследователь.** Опиши машину **вектором** [год, пробег, объём, цена] — и ближайшие соседи по этому вектору могут оказаться другой марки, но «с теми же ощущениями» и дешевле. Причём «похоже» зависит от того, **как мерить**: евклидово расстояние и косинус дадут разные списки соседей.

### Ваша задача
- Составьте вектор признаков машины и найдите k ближайших соседей к выбранной.
- Сравните два определения «похожести»: евклидово расстояние и косинусную близость.
- Предложите 3 реальные альтернативы «как Camry 2015, но доступнее».

### 🧮 Какой матаппарат стоит попробовать (и зачем именно он)
- **Вектор признаков + норма.** Машина как точка в пространстве характеристик.
- **z-нормировка — обязательна, а не для красоты.** Год ≈ 2015, пробег ≈ 150 000, объём ≈ 2, цена ≈ 1 000 000 — шкалы несопоставимы; без нормировки «похожесть» определит один пробег (да ещё и мусор в нём). ⚠️ Сначала уберите мусорный пробег и цены.
- **Евклидово расстояние** — «близко по всем признакам сразу».
- **Скалярное произведение → косинус** — «похоже по *профилю*/направлению», масштаб не важен. Поэкспериментируйте: включать ли цену в вектор?
- **kNN вручную** — отсортировать по расстоянию/близости и взять top-k.

### 📊 Что построить
- Таблица top-5 соседей по евклиду **и** по косинусу — где списки расходятся.
- 2D-проекция (например, год×цена или пробег×цена) с подсветкой цели и её соседей.
- Столбик цен: цель против найденных альтернатив.

### 🎤 Как защищать
Покажите 3 альтернативы Camry с ценами и на сколько % они доступнее. Объясните, почему евклид и косинус выбрали разное — и какой смысл «похожести» ближе к запросу.

*Прежде чем писать код — впишите сюда свой наивный ответ одной строкой:* `...`

In [ ]:
# --- T3: ваш код. План — в задании выше. ------------------------------------
# Подсказка по шагам:
#   1) FEATS = ["year_n","mileage_n","engine_n","price_kgs"]   # выбор за вами
#   2) sub = df.dropna(subset=FEATS)
#      sub = sub[sub.mileage_n.between(1,500000) & sub.price_kgs.between(50000,5e6)]  # чистка
#   3) z-нормировка: Z = (sub[FEATS]-sub[FEATS].mean())/sub[FEATS].std()
#   4) target = вектор выбранной машины (напр. Toyota Camry 2015) в Z
#   5) евклид: np.linalg.norm(Z.values - target, axis=1)
#      косинус: (Z @ target)/(np.linalg.norm(Z,axis=1)*np.linalg.norm(target))
#   6) argsort -> top-5 по каждой метрике, сравните списки

---
# 🧩 T4. 1000 просмотров, а не звонят

> Ты выставил свою машину. Одни объявления собирают тысячи просмотров, но телефон молчит; другие — сотню просмотров и шквал звонков. Что делает объявление продающим?

**🙂 Наивный взгляд.** Чем больше просмотров — тем лучше. И надо взять платную VIP-подсветку.

**🔬 А что видит исследователь.** Просмотры ≠ звонки. Важна **конверсия** — сколько просмотров превращается в звонки. Есть «магниты просмотров», по которым не звонят; бывает, что заниженная цена гонит звонки (часто это приманка). А платный VIP поднимает просмотры, но конверсию — почти нет.

### Ваша задача
- Посмотрите, как связаны просмотры и звонки, и где связь ломается.
- Введите меру «эффективности» объявления и найдите, кто конвертит хорошо, а кто лишь копит просмотры.
- Проверьте: VIP-подсветка реально помогает продать — или только накручивает просмотры?

### 🧮 Какой матаппарат стоит попробовать (и зачем именно он)
- **Признак-отношение `conv_rate` = звонки / просмотры.** Иногда честнее сырых чисел — уже посчитан за вас.
- **2D-облако (просмотры, звонки) + центроиды.** Нужно, чтобы увидеть «магниты просмотров» (точки вправо-вниз).
- **Сравнение групп боксплотом:** VIP vs обычные и по ценовым квартилям — что реально двигает звонки. Ключ: сравните **и просмотры, и конверсию**.
- **Медиана vs среднее** конверсии (пара вирусных объявлений перекосит среднее).

### 📊 Что построить
- Scatter `views`–`callers_count` с линией «типичной конверсии».
- Боксплот `conv_rate` для VIP против обычных.
- `conv_rate` по ценовым квартилям — гонит ли низкая цена звонки?

### 🎤 Как защищать
Сформулируйте рецепт продающего объявления и честный ответ: стоит ли платить за VIP. Подкрепите цифрой.

*Прежде чем писать код — впишите сюда свой наивный ответ одной строкой:* `...`

In [ ]:
# --- T4: ваш код. План — в задании выше. ------------------------------------
# Подсказка по шагам:
#   1) sub = df[df.views > 0]                       # conv_rate уже посчитан
#   2) sns.scatterplot(sub, x="views", y="callers_count", alpha=0.2)
#   3) VIP: sub.groupby("is_vip")[["views","conv_rate"]].median()  # и просмотры, и конверсия
#   4) по цене: sub["pq"] = pd.qcut(sub.price_kgs, 4)
#      sub.groupby("pq", observed=True)["conv_rate"].median()
#   5) осторожно с выбросами/шкалой

---
# 🧩 T5. Автосалоны и мусор

> Доска забита автосалонами и мусором: одни аккаунты вешают под 200 объявлений, у кого-то пробег «1 111 111 111 км», а цена — «1 сом». Вычисли их и оцени, как они портят «среднюю цену рынка».

**🙂 Наивный взгляд.** Все объявления — живые частные машины. Дубли заголовков = спам, их и будем ловить.

**🔬 А что видит исследователь.** Дубли заголовков — **ложный след**: заголовки авто генерируются из характеристик, поэтому совпадают у разных машин. Настоящие маркеры — **много объявлений с одного аккаунта** и **невозможные значения** (пробег в миллиард, цена в 1 сом). «Автосалон» — это не готовая метка, а признак, который надо собрать из улик.

### Ваша задача
- Найдите аномально активных продавцов — кандидатов в автосалоны.
- Соберите из улик простой «индекс подозрительности» и разметьте объявления.
- Покажите, как мусор ломает «среднюю цену» (сравните mean и median до/после чистки).

### 🧮 Какой матаппарат стоит попробовать (и зачем именно он)
- **Распределение объявлений на продавца (log-шкала) + перцентиль.** ⚠️ IQR тут снова вырождается: у большинства ровно 1 объявление (Q1=Q3=1), порог «>1» бессмыслен. Берём высокий перцентиль (топ-1%) или явный разрыв (у салонов 180–196 объявлений).
- **Вектор-улика из z-оценок** (`n_ads_in_data`, экстремальность пробега и цены) → индекс подозрительности = сумма/длина компонент.
- **Перцентили** — чтобы отсечь невозможные значения пробега и цены.
- **Среднее vs медиана** на грязных и очищенных данных — во сколько раз мусор задрал среднее.

### 📊 Что построить
- Гистограмма `n_ads_in_data` в log-шкале — где обрывается «частник» и начинается «салон».
- Отмеченные подозрительные объявления (сколько нашли).
- Средняя vs медианная цена до и после чистки — наглядный удар по «среднему».

### 🎤 Как защищать
Оцените, сколько на доске салонов и мусора и насколько они врут о «средней цене рынка». Покажите 2–3 самых наглых объявления, которые ваш индекс поймал.

*Прежде чем писать код — впишите сюда свой наивный ответ одной строкой:* `...`

In [ ]:
# --- T5: ваш код. План — в задании выше. ------------------------------------
# Подсказка по шагам:
#   1) активность продавцов: df["n_ads_in_data"].describe()
#      гистограмма в log-шкале: plt.hist(...); plt.yscale("log")
#   2) порог "салона": высокий перцентиль (напр. .99) или явный разрыв
#      (правило 1.5*IQR тут вырождается — посмотрите на квартили, поймёте почему)
#   3) индекс подозрительности: сумма z-оценок улик
#      (n_ads_in_data, насколько экстремальны mileage_n и price_kgs ...)
#   4) mean vs median цены до и после удаления мусора
#   5) top пойманных: df.nlargest(...)[["title","price_kgs","mileage_n","n_ads_in_data"]]

---
## 🎤 Как оформить защиту (памятка)

Выступление — 3 минуты. Не пересказ кода, а **история одного решения**:

1. **Вопрос кейса** одной фразой + ваш **наивный ответ** (тот, что записали в начале).
2. **Что показали данные.** 3–5 графиков. Каждый график отвечает на вопрос «и что?».
3. **Поворот.** Где интуиция обманула? Один слайд «казалось → оказалось».
4. **Вердикт.** Конкретное решение и цифры. Не «Toyota дороже», а «за 500 тыс. сом честно берётся вот это, и вот почему».
5. **Честность.** Что осталось за кадром, где данные слабые, чему бы вы не доверяли.

**Чек-лист хорошего графика:** есть заголовок; подписаны оси; видно единицы (сомы, км, л, год); нет «каши»; **виден вывод**, а не просто «данные».

> Помните правило дома: комментарии в коде — по-английски, тексты и подписи — по-русски.

Удачи. Данные грязные — тем ценнее чистый вывод. 🚗💨